# Análisis Descriptivo y Estadístico con interpretaciones.
## ATUS_Anual_2024
### Proyecto Final

Este notebook que elegimos estudia los registros de accidentes de tránsito terrestre publicados por INEGI durante 2024. 
A continuación se hará un análisis exhaustivo del dataset en limpio con el fin de extraer la información más relevante de estos conjuntos y poder dar conclusiones acerca de estos.


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go # Importamos la biblioteca plotly.graph_objects para crear gráficos personalizados
import numpy as np # Importamos la biblioteca numpy para realizar operaciones numéricas
df_original = pd.read_csv("../data/atus_anual_2024_limpio.csv")
df = df_original.copy()

### Localización
Con el fin de poder enfocar nuestro análisis en un fenómeno relevante en nuestros datos, es importante generar un panorama general de estos que nos puedan dar una evidencia de dichos eventos que estamos buscando. Exponemos este panorama en una presentación de los datos predominantes de categorías que nos ayudan a entender el motivo de los accidentes de transito, el tema principal de nuestro dataset.
Fue necesario implementar una columna adicional para mapear el ID de las entidades federativas para poder ser desplegadas de manera correcta en nuestras gráficas.

In [2]:
# Edades más frecuentes de los conductores involucrados
edad_moda = df['ID_EDAD'].mode()[0]
print(f"La edad más frecuente de los conductores involucrados es: {int(edad_moda)} años.")

# géneros más frecuentes involucrados en los accidentes 
sexo_moda = df['SEXO'].mode()[0]
print(f"El género con mayor reincidencia en los siniestros es: {sexo_moda}.")

# Diccionario oficial de IDs numéricos a nombres de Entidades Federativas en México
diccionario_estados = {
    1: 'Aguascalientes', 2: 'Baja California', 3: 'Baja California Sur', 
    4: 'Campeche', 5: 'Coahuila', 6: 'Colima', 7: 'Chiapas', 
    8: 'Chihuahua', 9: 'Ciudad de México', 10: 'Durango', 
    11: 'Guanajuato', 12: 'Guerrero', 13: 'Hidalgo', 14: 'Jalisco', 
    15: 'Estado de México', 16: 'Michoacán', 17: 'Morelos', 18: 'Nayarit', 
    19: 'Nuevo León', 20: 'Oaxaca', 21: 'Puebla', 22: 'Querétaro', 
    23: 'Quintana Roo', 24: 'San Luis Potosí', 25: 'Sinaloa', 
    26: 'Sonora', 27: 'Tabasco', 28: 'Tamaulipas', 29: 'Tlaxcala', 
    30: 'Veracruz', 31: 'Yucatán', 32: 'Zacatecas'
}
# Creamos una columna auxiliar con el nombre mapeado
df['NOMBRE_ENTIDAD'] = df['ID_ENTIDAD'].map(diccionario_estados)

# Entidad con más accidentes
entidad_moda = df['NOMBRE_ENTIDAD'].mode()[0]
print(f"La entidad con mayor frecuencia de accidentes es: {entidad_moda}.")

# Vehículos más involucrados en accidentes (Sumando las banderas de tipo de vehículo)
columnas_vehiculos = ['AUTOMOVIL', 'CAMPASAJ', 'MICROBUS', 'PASCAMION', 'OMNIBUS', 
                      'TRANVIA', 'CAMIONETA', 'CAMION', 'TRACTOR', 'FERROCARRI', 
                      'MOTOCICLET', 'BICICLETA', 'OTROVEHIC']
frecuencia_vehiculos = df[columnas_vehiculos].sum().sort_values(ascending=False)
vehiculo_moda = frecuencia_vehiculos.index[0]
print(f"El tipo de vehículo que más sufre accidentes es: {vehiculo_moda}.")

hora_moda = df['ID_HORA'].mode()[0]
print(f"Hora en la que más ocurren accidentes: {int(hora_moda)}.")

La edad más frecuente de los conductores involucrados es: 30 años.
El género con mayor reincidencia en los siniestros es: Hombre.
La entidad con mayor frecuencia de accidentes es: Nuevo León.
El tipo de vehículo que más sufre accidentes es: AUTOMOVIL.
Hora en la que más ocurren accidentes: 14.


De lo anterior se puede concluir que la mayor parte de las personas involucradas en accidentes de tránsito son varones, las víctimas cuentan con 30 años y la mayor parte de estos accidentes envuelven problemas con automóviles de uso personal. 
Como veremos a continuación con la comparación de estos datos predominantes frente al resto del conjunto, cada uno de estos datos tiene mucha más probabilidad de ocurrir que todos los demás. Las siguientes gráficas presentan el contraste entre cada una de las categorías y sus conjuntos de datos.

In [3]:

# GÉNERO DE LAS VÍCTIMAS
df_sexo = df['SEXO'].value_counts().reset_index()
df_sexo.columns = ['Género', 'Total']

fig_sexo = px.pie(df_sexo, values='Total', names='Género',
                  title='Distribución de Accidentes por Género',
                  color_discrete_sequence=px.colors.sequential.RdBu)
fig_sexo.update_traces(textinfo='percent+label')
fig_sexo.show()

In [4]:
# TOP 10 ENTIDADES CON MÁS ACCIDENTES
top_entidades = df['NOMBRE_ENTIDAD'].value_counts().head(10).reset_index()
top_entidades.columns = ['NOMBRE_ENTIDAD', 'Cantidad de Accidentes']
top_entidades['NOMBRE_ENTIDAD'] = top_entidades['NOMBRE_ENTIDAD'].astype(str)

fig_entidades = px.bar(top_entidades, x='Cantidad de Accidentes', y='NOMBRE_ENTIDAD',
                       orientation='h',
                       title='Top 10 Entidades Federativas con Mayor Frecuencia de Accidentes',
                       labels={'NOMBRE_ENTIDAD': 'Nombre de la Entidad'},
                       color='Cantidad de Accidentes', color_continuous_scale='Cividis')

fig_entidades.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_white')
fig_entidades.show()

In [5]:
#  DISTRIBUCIÓN POR TIPO DE VEHÍCULO (Gráfico de Pastel)
df_vehiculos = frecuencia_vehiculos.reset_index()
df_vehiculos.columns = ['Tipo de Vehículo', 'Total de Incidentes']

fig_vehiculos = px.pie(df_vehiculos, values='Total de Incidentes', names='Tipo de Vehículo',
                       title='Proporción de Accidentes por Tipo de Vehículo',
                       color_discrete_sequence=px.colors.qualitative.Prism)
fig_vehiculos.update_traces(textinfo='percent')
fig_vehiculos.show()


In [6]:
#Aseguramos que la columna ALIENTO esté limpia y estandarizada
df['ALIENTO_LIMPIO'] = df['ALIENTO'].astype(str).str.strip().str.capitalize()
df['ALIENTO_LIMPIO'] = df['ALIENTO_LIMPIO'].replace({'Si': 'Sí'})
df_aliento_filtrado = df[df['ALIENTO_LIMPIO'].isin(['Sí', 'No'])]

# 2. Agrupamos y contamos las frecuencias
df_pastel = df_aliento_filtrado['ALIENTO_LIMPIO'].value_counts().reset_index()
df_pastel.columns = ['Estatus Aliento', 'Cantidad']

# Generar la gráfica
fig_pastel = px.pie(df_pastel, 
                    names='Estatus Aliento', 
                    values='Cantidad',
                    title='Proporción de Accidentes según la Presencia de Aliento Alcohólico',
                    hole=0.4, # Convierte el pastel en una dona
                    color='Estatus Aliento',
                    # Asignamos colores institucionales: Gris suave para el No, Rojo Alerta para el Sí
                    color_discrete_map={'No': '#BDC3C7', 'Sí': '#E74C3C'})

# Ajustamos las etiquetas para mostrar tanto el conteo real como el porcentaje exacto
fig_pastel.update_traces(textinfo='percent+value', textposition='inside',
                         marker=dict(line=dict(color='#FFFFFF', width=2)))

fig_pastel.update_layout(template='plotly_white')
fig_pastel.show()

Pero hay dos columnas de información que también nos dan información adicional que consideramos relevante para un análisis más exhaustivo: La edad y las horas en las que ocurren los accidentes. Podemos determinar un promedio de la edad de los involucrados en los accidentes, así como el poder recopilar una media en la que generalmente los accidentes, así como recopilar un punto medio de la información de ambas categorías conocido como mediana.

In [7]:
# Limpieza estricta de edad
df['ID_EDAD'] = pd.to_numeric(df['ID_EDAD'], errors='coerce')
df['ID_EDAD'] = df['ID_EDAD'].replace([0, 99], np.nan)

# Estadísticos
edad_media = df['ID_EDAD'].mean()
edad_mediana = df['ID_EDAD'].median()

#  Agrupamos por edad pero ordenamos numéricamente el eje X
df_edades_completo = df['ID_EDAD'].value_counts().reset_index()
df_edades_completo.columns = ['Edad', 'Cantidad de Accidentes']
df_edades_completo = df_edades_completo.sort_values(by='Edad') # Orden numérico lineal

# Gráfica de distribución
fig_edad_completa = px.bar(df_edades_completo, x='Edad', y='Cantidad de Accidentes',
                           title=f'Distribución Completa de Accidentes por Edad (Líneas de Tendencia Central)',
                           labels={'Edad': 'Edad Real del Conductor'},
                           color='Cantidad de Accidentes', color_continuous_scale='viridis')

# AGREGAR LÍNEAS VERTICALES
fig_edad_completa.add_vline(x=edad_media, line_width=3, line_dash="dash", line_color="red",
                            annotation_text=f"Media ({edad_media:.1f} años)", annotation_position="top right")

fig_edad_completa.add_vline(x=edad_mediana, line_width=3, line_dash="dot", line_color="green",
                            annotation_text=f"Mediana ({edad_mediana:.1f} años)", annotation_position="top left")

fig_edad_completa.update_layout(template='plotly_white')
fig_edad_completa.show()

In [8]:
# Horas con más accidentes (Gráfico de Barras)
# Cálculo de las horas (media y mediana)
hora_media = df['ID_HORA'].mean()
hora_mediana = df['ID_HORA'].median()
df_horas = df['ID_HORA'].value_counts().reset_index()
df_horas.columns = ['Hora', 'Cantidad de Accidentes']
df_horas = df_horas.sort_values(by='Hora')

fig_hora = px.bar(df_horas, x='Hora', y='Cantidad de Accidentes',
                  title=f'Frecuencia de Accidentes por Hora (Media: {hora_media:.1f}h | Mediana: {hora_mediana:.1f}h)',
                  labels={'Hora': 'Hora del Día (Formato 24h)'},
                  color='Cantidad de Accidentes', color_continuous_scale='Cividis')

# Añadir línea de la Media (Promedio)
fig_hora.add_vline(x=hora_media, line_width=3, line_dash="dash", line_color="red",
                   annotation_text=f"Media ({hora_media:.1f})", annotation_position="top right")

# Añadir línea de la Mediana
fig_hora.add_vline(x=hora_mediana, line_width=3, line_dash="dot", line_color="green",
                   annotation_text=f"Mediana ({hora_mediana:.1f})", annotation_position="top left")

fig_hora.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1), template='plotly_white')
fig_hora.show()

### Variabilidad
Las dos últimas gráficas que se presentaron dan evidencia para una conclusión interesante para las categorías de edad y horas en las que ocurren los accidentes, teniendo la posibilidad de dar paso a una conclusión al rango de edades/horas predominantes en nuestros datos; estas conclusiones deben de ser respaldadas por cálculos. Es aquí donde las medidas de variabilidad entran en juego, mediante las medidas IQR, varianza y desviación estándar respaldaremos una conclusión respecto al rango de edad en los involucrados, así como dar un horario en donde es más probable que ocurran accidentes. 

In [9]:
# Aseguramos que los datos estén limpios y en el formato correcto. 0 es un valor que indica que el perpetrador se fugó, y 99 tiene edad desconocida. 
df['ID_HORA'] = pd.to_numeric(df['ID_HORA'], errors='coerce')

print("=========================================================")
print("             MEDIDAS DE VARIABILIDAD (ANÁLISIS)          ")
print("=========================================================")

# Cálculo de métricas de variabilidad para categorías numéricas.
def analizar_dispersion(dataframe, columna, nombre_variable):
    # Pandas ignora los NaNs automáticamente en estos cálculos
    v_min = dataframe[columna].min()
    v_max = dataframe[columna].max()
    rango = v_max - v_min
    varianza = dataframe[columna].var()
    desviacion = dataframe[columna].std()
    
    q1 = dataframe[columna].quantile(0.25)
    q3 = dataframe[columna].quantile(0.75)
    iqr = q3 - q1
    
    print(f"Métricas de Dispersión para: {nombre_variable}")
    print(f"  - Rango Absoluto:       {rango:.2f} (Desde {v_min} hasta {v_max})")
    print(f"  - Rango Intercuartílico (IQR): {iqr:.2f} (Q1: {q1} | Q3: {q3})")
    print(f"  - Varianza:            {varianza:.2f}")
    print(f"  - Desviación Estándar: {desviacion:.2f}")
    print("-" * 50)
    return q1, q3, iqr

# Ejecución de los cálculos numéricos
q1_e, q3_e, iqr_e = analizar_dispersion(df, 'ID_EDAD', 'EDAD DEL CONDUCTOR')
q1_h, q3_h, iqr_h = analizar_dispersion(df, 'ID_HORA', 'HORA DEL ACCIDENTE')

             MEDIDAS DE VARIABILIDAD (ANÁLISIS)          
Métricas de Dispersión para: EDAD DEL CONDUCTOR
  - Rango Absoluto:       86.00 (Desde 12.0 hasta 98.0)
  - Rango Intercuartílico (IQR): 21.00 (Q1: 26.0 | Q3: 47.0)
  - Varianza:            213.42
  - Desviación Estándar: 14.61
--------------------------------------------------
Métricas de Dispersión para: HORA DEL ACCIDENTE
  - Rango Absoluto:       23.00 (Desde 0 hasta 23)
  - Rango Intercuartílico (IQR): 9.00 (Q1: 9.0 | Q3: 18.0)
  - Varianza:            34.31
  - Desviación Estándar: 5.86
--------------------------------------------------


Las medidas de variabilidad nos dan más detalles de la población de estas categorías, analicemos primero las edades.
El rango en está categoría nos revela cuál es la edad más temprana y tardía encontrada en los conductores, este caso nos revela que el conductor más joven involucrado en un accidente tenía 12 años de edad, mientras que la persona más longeva involucrada en un accidente de tránsito contaba con 98 años de edad. Mientras tanto, el rango en las horas solo nos demuestra que los accidentes pueden ocurrir a cualquier hora.
El rango de edades en los conductores puede ser un resultado preocupante, pero el indice de IQR puede demostrar que estos casos son atípicos, la varianza y la desviación estándar funcionan de la misma manera al indicar de que forma tienden a variar nuestros casos, esto lo podemos ver en las siguientes gráficas de BoxPlots que indican donde se concentran nuestros resultados al mismo tiempo que mediante los Whiskers/bigotes se determinan donde usualmente los casos de edad y horas de accidentes terminan.

In [10]:
# Boxplot para Edad
fig_box_edad = px.box(df, y='ID_EDAD', 
                      title='Análisis de Valores Atípicos (Outliers) en la Edad de Conductores',
                      labels={'ID_EDAD': 'Edad Real (Años)'},
                      color_discrete_sequence=['#4A90E2'])
fig_box_edad.update_layout(template='plotly_white')
fig_box_edad.show()

# Boxplot para Horas
fig_box_hora = px.box(df, y='ID_HORA', 
                      title='Distribución de Tiempos y Valores Atípicos en las Horas de Siniestros',
                      labels={'ID_HORA': 'Hora del Día (0-23)'},
                      color_discrete_sequence=['#E67E22'])
fig_box_hora.update_layout(template='plotly_white')
fig_box_hora.show()

Ambas cajas nos dan información bastante contundente: No es poco común, es incluso frecuente que menores de edad no solo conduzcan, sino que se vean envueltos en accidentes de tránsito; pero también es cierto que casos donde el conductor tenga una edad superior a los 79 años son muy singulares, por lo cuál sería equivocado tomarlo como una tendencia y sería correcto tartar estas edades como casos aislados.
De la caja de horas también obtenemos una conclusión muy importante: No existe una hora o periodo de tiempo en los que no sucedan accidentes de manera frecuente. 
Pero la desviación estándar también nos indica donde nuestros registros suelen fuertemente permanecer, permitiéndonos dar una generalidad acerca de los rangos de edad de los conductores que suelen tener accidentes, así como un horario en donde ocurren más accidentes.

In [11]:
# Cálculo de la desviación estándar para ambas variables
edad_std = df['ID_EDAD'].std()
hora_std = df['ID_HORA'].std()
# Histograma normalizado por densidad
fig_densidad_edad = px.histogram(df, x='ID_EDAD', histnorm='probability density',
                                 title=f'Visualización de la Variabilidad: Dispersión de Edades',
                                 labels={'ID_EDAD': 'Edad al momento del siniestro'},
                                 color_discrete_sequence=['#A6C8E0'], nbins=50)

# Línea vertical de la Media
fig_densidad_edad.add_vline(x=edad_media, line_width=2, line_dash="dash", line_color="red",
                            annotation_text=f"Media ({edad_media:.1f})", annotation_position="top right")

# Sombreado del área de 1 Desviación Estándar (Media - STD a Media + STD)
fig_densidad_edad.add_vrect(x0=edad_media - edad_std, x1=edad_media + edad_std, 
                            fillcolor="rgba(128, 128, 128, 0.15)", line_width=0,
                            annotation_text=f"1 Desviación (±{edad_std:.1f} años)", 
                            annotation_position="top left")

fig_densidad_edad.update_layout(template='plotly_white', yaxis_title='Densidad de Casos', showlegend=False)
fig_densidad_edad.show()
### Histograma normalizado por densidad para Horas
fig_densidad_hora = px.histogram(df, x='ID_HORA', histnorm='probability density',
                                 title=f'Visualización de la Variabilidad: Dispersión de Horarios',
                                 labels={'ID_HORA': 'Hora del Siniestro (0-23 hrs)'},
                                 color_discrete_sequence=['#F3C68F'], nbins=24)

# Línea vertical de la Media
fig_densidad_hora.add_vline(x=hora_media, line_width=2, line_dash="dash", line_color="red",
                            annotation_text=f"Media ({hora_media:.1f}h)", annotation_position="top right")

# Sombreado del área de 1 Desviación Estándar para las Horas
fig_densidad_hora.add_vrect(x0=max(0, hora_media - hora_std), x1=min(23, hora_media + hora_std), 
                            fillcolor="rgba(128, 128, 128, 0.15)", line_width=0,
                            annotation_text=f"1 Desviación (±{hora_std:.1f} hrs)", 
                            annotation_position="top left")

fig_densidad_hora.update_layout(template='plotly_white', yaxis_title='Densidad de Casos',
                                xaxis=dict(tickmode='linear', tick0=0, dtick=2), showlegend=False)
fig_densidad_hora.show()

O sea, basado en los resultados de las gráficas y la desviación estándar, es válido hacer los siguientes estamentos:
"La mayor parte de los accidentes de tránsito envuelven a conductores de entre 22 y 52 años de edad"
"La mayoría de los accidentes ocurren en un horario de entre 7 de la mañana hasta las 7 de la noche"

### Heterogeneidad.
Tenemos conocimiento acerca de las entidades que más presentan accidentes de tránsito en todo el país, pero de momento con base a nuestros datos no tenemos una prueba la cuál nos indique que los accidentes de tránsito sean un problema generalizado. Además, no sabemos si los accidentes están distribuidos de forma similar cada día de la semana o si existirá algún día que por una razón u otra presente una mayor incidencia de accidentes. Mediante el índice de impureza de Gini y la medida de entropía podemos determinar si existe una tendencia en alguno de estos campos que podamos explotar.
A la entropía se le define como el nivel de certeza que tendremos de que un resultado pertenezca a una categoría, mientras mayor certeza tengamos de que un registro pertenezca a un conjunto de datos, con menor entropía contamos.
Recordemos que el indice de impureza de Gini y el indice de entropía de Shannon evalúan el mismo fenómeno de distinta forma, o sea, que a pesar de que ambos no den el mismo resultado, son directamente proporcionales.

In [12]:
# Los días de la semana se presentan como números del 1 al 7, donde 1 es Lunes y 7 es Domingo. Creamos una columna auxiliar con el nombre mapeado.
diccionario_dias = {
    1: 'Lunes', 2: 'Martes', 3: 'Miércoles', 4: 'Jueves', 
    5: 'Viernes', 6: 'Sábado', 7: 'Domingo',
}
if 'DIASEMANA' in df.columns:
    df['NOM_DIASEMANA'] = df['DIASEMANA'].map(diccionario_dias).fillna(df['DIASEMANA'])

# Función para calcular índices de heterogeneidad (Entropía de Shannon y Gini) para una serie de datos categóricos.
def calcular_indices_heterogeneidad(serie_datos, nombre_analisis):
    # Obtener frecuencias relativas (probabilidades p_i)
    frecuencias = serie_datos.value_counts(normalize=True)
    pi = frecuencias.values
    
    # Entropía de Shannon: -sum(p_i * log2(p_i))
    entropia = -np.sum(pi * np.log2(pi + 1e-12)) # 1e-12 evita el log(0)
    # Entropía Máxima Posible = log2(K) donde K es el número de categorías
    entropia_max = np.log2(len(pi))
    entropia_normalizada = entropia / entropia_max if entropia_max > 0 else 0
    
    # Índice de Gini (Impureza/Concentración): 1 - sum(p_i^2)
    # Nota: Para distribución uniforme, Gini_max = 1 - (1/K)
    gini_concentracion = 1 - np.sum(pi**2)
    
    print(f"=== ANÁLISIS DE HETEROGENEIDAD PARA: {nombre_analisis} ===")
    print(f"  - Clases/Categorías detectadas: {len(pi)}")
    print(f"  - Entropía de Shannon:          {entropia:.4f} (Máx teórica: {entropia_max:.4f})")
    print(f"  - Entropía Normalizada:         {entropia_normalizada:.4f} (0 = Concentración total, 1 = Uniforme)")
    print(f"  - Índice de Gini:               {gini_concentracion:.4f}")
    print("-" * 60)
    
    return frecuencias.reset_index()

# Ejecución de cálculos
df_freq_entidades = calcular_indices_heterogeneidad(df['NOMBRE_ENTIDAD'].dropna(), "ENTIDADES FEDERATIVAS")
df_freq_dias = calcular_indices_heterogeneidad(df['NOM_DIASEMANA'].dropna(), "DÍAS DE LA SEMANA")

=== ANÁLISIS DE HETEROGENEIDAD PARA: ENTIDADES FEDERATIVAS ===
  - Clases/Categorías detectadas: 32
  - Entropía de Shannon:          4.4374 (Máx teórica: 5.0000)
  - Entropía Normalizada:         0.8875 (0 = Concentración total, 1 = Uniforme)
  - Índice de Gini:               0.9290
------------------------------------------------------------
=== ANÁLISIS DE HETEROGENEIDAD PARA: DÍAS DE LA SEMANA ===
  - Clases/Categorías detectadas: 7
  - Entropía de Shannon:          2.8056 (Máx teórica: 2.8074)
  - Entropía Normalizada:         0.9994 (0 = Concentración total, 1 = Uniforme)
  - Índice de Gini:               0.8568
------------------------------------------------------------


En ambos casos podemos notar tanto un índice de entropía muy alto, como una impureza igualmente alta (ambas casi rozando el máximo nivel de entropía); retomando la definición anterior, se tiene poca certeza de que un registro cualquiera pertenezca a una entidad en específico, lo mismo con el día de la semana en el que ocurre el accidente, la siguiente gráfica muestra como las probabilidades de pertenecer a una categoría en específico son bastante similares

In [13]:
# Obtenemos la proporción relativa (de 0 a 1) para cada estado
df_entidades = df['NOMBRE_ENTIDAD'].dropna().value_counts(normalize=True).reset_index()
df_entidades.columns = ['Entidad', 'Proporción']

# Número de categorías para entidades (Normalmente 32)
k_entidades = len(df_entidades)
valor_uniforme_entidades = 1.0 / k_entidades

# Generamos la gráfica interactiva de Entidades
fig_uniformidad_entidad = px.bar(df_entidades, x='Entidad', y='Proporción',
                                 title='Probabilidad por Estado vs. Uniformidad Perfecta (Análisis de Entropía)',
                                 labels={'Proporción': 'Proporción (Proportion)', 'Entidad': 'ENTIDAD'},
                                 color_discrete_sequence=['#B22222']) # Color rojo similar a tu imagen

# Añadimos la línea horizontal discontinua teórica
fig_uniformidad_entidad.add_hline(y=valor_uniforme_entidades, 
                                  line_width=2, line_dash="dash", line_color="black",
                                  annotation_text=f"Distribución Uniforme Teórica ({valor_uniforme_entidades:.4f})", 
                                  annotation_position="top right")

fig_uniformidad_entidad.update_layout(template='plotly_white', xaxis_tickangle=-90)
fig_uniformidad_entidad.show()

In [14]:

# Convertimos a string, quitamos espacios en blanco y aseguramos formato tipo Título 
df['NOM_DIASEMANA'] = df['NOM_DIASEMANA'].astype(str).str.strip().str.capitalize()

# Diccionario de homologación para capturar variantes comunes sin acento
mapa_acentos = {
    'Miercoles': 'Miércoles',
    'Sabado': 'Sábado',
    'Domingo': 'Domingo',
    'Lunes': 'Lunes',
    'Martes': 'Martes',
    'Jueves': 'Jueves',
    'Viernes': 'Viernes'
}
df['NOM_DIASEMANA'] = df['NOM_DIASEMANA'].replace(mapa_acentos)

# Filtramos valores vacíos o cadenas de texto nulas que puedan quedar
df_dias_limpio = df[df['NOM_DIASEMANA'].notna() & (df['NOM_DIASEMANA'] != 'Nan')]

# Calculamos las proporciones 
df_dias = df_dias_limpio['NOM_DIASEMANA'].value_counts(normalize=True).reset_index()
df_dias.columns = ['Día', 'Proporción']
orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
df_dias['Día'] = pd.Categorical(df_dias['Día'], categories=orden_dias, ordered=True)
df_dias = df_dias.sort_values('Día')

# Número de categorías estricto (7 días de la semana)
k_dias = 7
valor_uniforme_dias = 1.0 / k_dias

# Generamos la gráfica 
fig_uniformidad_dias = px.bar(df_dias, x='Día', y='Proporción',
                              title='Probabilidad por Día de la Semana vs. Uniformidad Perfecta (Análisis de Entropía)',
                              labels={'Proporción': 'Proporción (Proportion)', 'Día': 'DÍA DE LA SEMANA'},
                              color_discrete_sequence=['#B22222'])

# Añadimos la línea horizontal discontinua teórica
fig_uniformidad_dias.add_hline(y=valor_uniforme_dias, 
                               line_width=2, line_dash="dash", line_color="black",
                               annotation_text=f"Distribución Uniforme Teórica ({valor_uniforme_dias:.4f})", 
                               annotation_position="top right")

fig_uniformidad_dias.update_layout(template='plotly_white')
fig_uniformidad_dias.show()

Notamos que la diferencia es mínima en los días de la semana, por lo que es seguro decir que el día de la semana no es exactamente un factor relevante para determinar cuando los accidentes son más propensos de ocurrir; aunque es notable destacar que los accidente tienen una mayor pero pequeña probabilidad de ocurrir entre los días Viernes y Sábado respecto a otros días . La gráfica de las entidades federativas fundamenta una afirmación poderosa: " A pesar de que algunas entidades federativas presentan una alta o una baja probabilidad relativa de hospedar accidentes, la probabilidad de que ocurra en un estado cualquiera es lo suficientemente grande para concluir que los accidentes de tránsito son un problema que asolan a todos los estados de una manera más o menos igual. 

Con el afán de obtener alguna otra conclusión interesante, hacemos un proceso similar al anterior sobre los meses para ver si algún mes tiene mayor probabilidad de tener más accidentes que los demás. 

In [15]:
# 1. Diccionario oficial para mapear los números de mes a nombres legibles
diccionario_meses = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto', 
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
# Aplicamos el diccionario
df['NOM_MES'] = df['MES'].map(diccionario_meses)
# Aplicamos la función de cálculo de índices de heterogeneidad para los meses del año
df_freq_meses = calcular_indices_heterogeneidad(df['NOM_MES'].dropna(), "MESES DEL AÑO")

=== ANÁLISIS DE HETEROGENEIDAD PARA: MESES DEL AÑO ===
  - Clases/Categorías detectadas: 12
  - Entropía de Shannon:          3.5837 (Máx teórica: 3.5850)
  - Entropía Normalizada:         0.9997 (0 = Concentración total, 1 = Uniforme)
  - Índice de Gini:               0.9165
------------------------------------------------------------


In [16]:
# Ajustamos nombres de columnas del resultado devuelto por tu función
df_freq_meses.columns = ['Mes', 'Proporción']

# Ordenamos cronológicamente de Enero a Diciembre para que el eje X tenga coherencia temporal
orden_meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 
               'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
df_freq_meses['Mes'] = pd.Categorical(df_freq_meses['Mes'], categories=orden_meses, ordered=True)
df_freq_meses = df_freq_meses.sort_values('Mes')

# Definimos el valor teórico de máxima entropía para 12 categorías (1/12)
k_meses = 12
valor_uniforme_meses = 1.0 / k_meses

# Generamos la gráfica interactiva en Plotly
fig_uniformidad_meses = px.bar(df_freq_meses, x='Mes', y='Proporción',
                               title='Probabilidad por Mes del Año vs. Uniformidad Perfecta (Análisis de Entropía)',
                               labels={'Proporción': 'Proporción (Proportion)', 'Mes': 'MES DEL AÑO'},
                               color_discrete_sequence=['#B22222']) # Mismo color institucional de tus gráficos

# Añadimos la línea horizontal discontinua teórica de uniformidad
fig_uniformidad_meses.add_hline(y=valor_uniforme_meses, 
                                line_width=2, line_dash="dash", line_color="black",
                                annotation_text=f"Distribución Uniforme Teórica ({valor_uniforme_meses:.4f})", 
                                annotation_position="top right")

fig_uniformidad_meses.update_layout(template='plotly_white')
fig_uniformidad_meses.show()

De acuerdo a la información del INEGI y nuestros cálculos, no existe una relación significativa entre la frecuencia de accidentes y el mes en el que acontecen.

### Concentración
Al tener altos índices de entropía en los meses, días y estados vamos a regresar a las variables que ya presentaron una variabilidad interesante con anterioridad, que son la edad de los conductores así como las horas en las que ocurren los siniestros, como hemos visto que existen un rango de datos que concentran la mayoría de registros de accidentes, veremos que tan fuerte es la ocurrencia de estos casos en los rangos anteriormente dados.
Para esto se emplea el indice de concentración, así como las gráficas conocidas como curvas de Lorenz que nos mostrará si los "horarios pico" tienen algún efecto en la ocurrencia de accidentes; así como la búsqueda de una conclusión solida que nos permita relacionar la edad con los horarios y conectar este fenómeno con actividades cotidianas como el trabajo, estudio u otros.


In [17]:
def calcular_lorenz_y_gini(serie_datos, nombre_variable):
    # Contamos frecuencias por categoría y ordenamos de menor a mayor volumen de accidentes
    conteos = serie_datos.value_counts().sort_values()
    n_categorias = len(conteos)
    
    #  Proporciones acumuladas de las categorías (Eje X de Lorenz)
    # Representa el % acumulado de "población" de categorías (ej. % de las 24 horas)
    eje_x = np.arange(1, n_categorias + 1) / n_categorias
    eje_x = np.insert(eje_x, 0, 0) # Empezar en el origen (0,0)
    
    # Proporciones acumuladas de los accidentes (Eje Y de Lorenz)
    # Representa el % acumulado de siniestros ocurridos
    proporciones_y = conteos.values / conteos.sum()
    eje_y = np.cumsum(proporciones_y)
    eje_y = np.insert(eje_y, 0, 0) # Empezar en el origen (0,0)
    
    # Cálculo del Coeficiente de Gini basado en el área bajo la curva
    # Gini = 1 - 2 * Área_Bajo_Lorenz
    area_bajo_curva = np.trapezoid(eje_y, eje_x)
    coeficiente_gini = 1 - 2 * area_bajo_curva
    
    return eje_x, eje_y, coeficiente_gini

# Ejecutamos los cálculos para Horas y Edades
x_hora, y_hora, gini_hora = calcular_lorenz_y_gini(df['ID_HORA'].dropna(), "Horas del Día")
x_edad, y_edad, gini_edad = calcular_lorenz_y_gini(df['ID_EDAD'].dropna(), "Edades de Conductores")
print(f"-> Coeficiente de Gini para las HORAS: {gini_hora:.4f}")
print(f"-> Coeficiente de Gini para las EDADES: {gini_edad:.4f}\n")

-> Coeficiente de Gini para las HORAS: 0.2346
-> Coeficiente de Gini para las EDADES: 0.5038



Podemos observar que el indice de concentración es bastante alto para las edades, lo cuál indica que unos pocos sectores de edad (los cuáles ya hemos analizado y recopilado) concentran la mayor cantidad de accidentes. A pesar de que la concentración en los horarios es un poco menor, es un número lo suficientemente grande como para concluir que existen algunas horas o periodos de tiempo que concentran más evidente. Esto lo podemos ver de forma más evidente en las curvas de Lorenz de ambas categorías.

In [18]:
# Grafica de Lorenz para las HORAS DEL DÍA
fig_lorenz_hora = go.Figure()

# Línea de Equidad Perfecta (Diagonal)
fig_lorenz_hora.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', 
                                     name='Equidad Perfecta (Distribución Uniforme)',
                                     line=dict(color='black', dash='dash')))
# Curva de Lorenz Real
fig_lorenz_hora.add_trace(go.Scatter(x=x_hora, y=y_hora, mode='lines+markers', 
                                     name=f'Curva de Lorenz (Gini: {gini_hora:.3f})',
                                     line=dict(color='firebrick', width=3)))

fig_lorenz_hora.update_layout(
    title=f'Curva de Lorenz de Accidentes por Hora del Día<br><sup>Índice de Concentración de Gini: {gini_hora:.4f}</sup>',
    xaxis_title='Porcentaje Acumulado de Horas del Día',
    yaxis_title='Porcentaje Acumulado de Accidentes',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig_lorenz_hora.show()

In [19]:
# Grafica de Lorenz para las EDADES DE LOS CONDUCTORES
fig_lorenz_edad = go.Figure()

# Línea de Equidad Perfecta (Diagonal)
fig_lorenz_edad.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', 
                                     name='Equidad Perfecta (Distribución Uniforme)',
                                     line=dict(color='black', dash='dash')))
# Curva de Lorenz Real
fig_lorenz_edad.add_trace(go.Scatter(x=x_edad, y=y_edad, mode='lines+markers', 
                                     name=f'Curva de Lorenz (Gini: {gini_edad:.3f})',
                                     line=dict(color='darkblue', width=3)))

fig_lorenz_edad.update_layout(
    title=f'Curva de Lorenz de Accidentes por Edad del Conductor<br><sup>Índice de Concentración de Gini: {gini_edad:.4f}</sup>',
    xaxis_title='Porcentaje Acumulado de Edades',
    yaxis_title='Porcentaje Acumulado de Accidentes',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig_lorenz_edad.show()

Con esto podemos confirmar algunas intuiciones hechas con algunas gráficas de la localización. En cuanto a las horas, podemos observar que la mayoría de accidentes ocurren en horas tempranas o medio día, mientras que las personas involucradas en accidentes son en su mayoría personas jóvenes o de mediana edad, esto nos da las evidencias suficientes para concluir que la mayoría de incidentes ocurren horas de más actividad en ciudades, donde personas, en su mayoría varones en edad de estudiar/trabajar tienen la necesidad de transportarse a realizar alguna de estas actividades. 

### Correlaciones
A lo largo de este análisis, hemos evitado operar bajo columnas que contienen información considerada relevante para el registro de accidentes: El número de heridos o personas fallecidas a causa de estas. El Dataset cuenta con las siguientes columnas que hacen el registro de estos casos:  'CONDMUERTO', 'CONDHERIDO', 'PASAMUERTO', 'PASAHERIDO', 'PEATMUERTO', 'PEATHERIDO', 'CICLMUERTO', 'CICLHERIDO', 'OTROMUERTO', 'OTROHERIDO', 'CLASACC', 'CONDUCTOR_FUGADO'. Las cuáles documentan si hubo Conductores, peatones, ciclistas u otras victimas en el accidente, también teniendo una columna que lleva el registro de conductores fugados después de un accidente. El motivo fue reservar estos resultado para el apartado de correlación, pues ya teniendo un análisis bastante completo de las variables "circunstanciales" de los accidentes ahora podemos relacionar las más relevantes que encontramos con el número de victimas (si es el caso).
La limpieza del dataset también nos permite hacer esto sin muchas preocupaciones, ya que la variable "CLASACC" actúa como una bandera que indica las repercusiones del accidente y así prevenir el ruido al no tomar en cuenta registros en accidentes fatales o no fatales.
Al ser columnas con valores numéricas, podemos directamente hacer un heatMap que contenga todas las columnas de nuestro dataset limpio que contengan valores numéricos, así podremos observar si existen relaciones entre variables en todo el contenido del dataset.

In [20]:
# Aseguramos la limpieza previa de Edad y Hora
df['ID_EDAD'] = pd.to_numeric(df['ID_EDAD'], errors='coerce').replace([0, 99], np.nan)
df['ID_HORA'] = pd.to_numeric(df['ID_HORA'], errors='coerce')

# 2. Convertimos la bandera de fuga y las víctimas a numérico estricto
df['CONDUCTOR_FUGADO'] = pd.to_numeric(df['CONDUCTOR_FUGADO'], errors='coerce').fillna(0).astype(int)

columnas_victimas = [
    'CONDMUERTO', 'CONDHERIDO', 'PASAMUERTO', 'PASAHERIDO', 
    'PEATMUERTO', 'PEATHERIDO', 'CICLMUERTO', 'CICLHERIDO', 
    'OTROMUERTO', 'OTROHERIDO'
]

for col in columnas_victimas:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
# Seleccionamos las columnas numéricas clave para la matriz
columnas_analisis = ['ID_EDAD', 'ID_HORA', 'CONDUCTOR_FUGADO'] + columnas_victimas
df_corr_temp = df[columnas_analisis].copy()


mediana_temporal = df_corr_temp['ID_EDAD'].median()
df_corr_temp['ID_EDAD'] = df_corr_temp['ID_EDAD'].fillna(mediana_temporal)

# Calculamos la matriz sobre el dataframe 
matriz_corr = df_corr_temp.corr(method='pearson')

# Visualización del Heatmap 
fig_heatmap = px.imshow(matriz_corr,
                        text_auto=".2f",
                        aspect="auto",
                        title="Matriz de Correlación",
                        color_continuous_scale='RdBu_r',
                        zmin=-1, zmax=1)

fig_heatmap.update_layout(template='plotly_white', width=900, height=700)
fig_heatmap.show()

En el anterior heaMap podemos ver algunas relaciones que se pueden considerar como obvias, como la relación entre conductores y pasajeros muertos/heridos, donde si uno de los dos se presenta muerto/herido, es probable que el otro también. Aunque a simple vista no se presenten otras relaciones fuertes entre la información, existe un fenómeno que merece la pena inspeccionar más de cerca.

#### Un fenómeno visto más a detalle
En el heatmap no es muy claro que realmente la gravedad del incidente afecta en la decisión de los conductores de abandonar la escena, pero si lo observamos a más profundidad, de acuerdo a nuestro análisis, la gravedad del siniestro si influye de una forma considerable a la decisión de fugarse como podemos ver en la siguiente gráfica. Mientras que el conductor decide permanecer en la escena del accidente cuando el incidente deja solo heridos o daños, es mucho más probable (casi un 20%) que un conductor decida abandonar la escena cuando hay victimas mortales.

In [21]:
# Filtramos valores nulos o raros de CLASACC para evitar ruido
df_severidad = df[df['CLASACC'].isin(['Fatal', 'No fatal', 'Sólo daños'])].copy()

# ¿Influye la severidad en la decisión de fugarse?
# Calculamos la tasa de fuga por tipo de accidente
tabla_fuga = pd.crosstab(df_severidad['CLASACC'], df_severidad['CONDUCTOR_FUGADO'], normalize='index') * 100
tabla_fuga = tabla_fuga.reset_index()
tabla_fuga.columns = ['Clasificación del Accidente', 'Se Quedó (%)', 'Se Fugó (%)']

fig_fuga = px.bar(tabla_fuga, x='Clasificación del Accidente', y=['Se Quedó (%)', 'Se Fugó (%)'],
                  title='Porcentaje de Conductores Fugados según la Gravedad del Accidente',
                  labels={'value': 'Porcentaje (%)', 'variable': 'Comportamiento'},
                  barmode='group',
                  color_discrete_sequence=['#2CA02C', '#D62728'])

fig_fuga.update_layout(template='plotly_white')
fig_fuga.show()


#### Búsqueda de correlaciones adicionales
Nuestra búsqueda de relaciones inicialmente resultó infructífera al no haber podido establecer de manera más clara. Por lo que realizamos una comparación entre el número de vehículos que estuvieron afectados en los accidentes y relacionarlos con las variables de mortandad previamente rescatadas para concluir si existe mayor riesgo de resultar herido o perder la vida si se presentan cierto tipo de vehículos.
Además, también analizamos dos banderas adicionales: ALIENTO y CINTURON. Los cuáles son otros factores de riesgo en los accidentes, ya que ALIENTO determina si una persona se presentaba en estado de ebriedad al momento de ocurrir el accidente o si los conductores llevaban puesto el cinturón o no al momento de ocurrir el siniestro, determinaremos si estos factores aumentan o disminuyen el riesgo de resultar herido o morir. 

In [22]:

# Preparación de datos para análisis de regresión logística (Predicción de Fuga)

# Hacemos la limpieza de las variables, ya que CINTURON y ALIENTO son dos factores de riesgo clave 
# que queremos incluir en el modelo, pero sus banderas de "No_Aplicable" o "Se ignora" pueden generar ruido.
# Transformación de ALIENTO (1 = Con aliento alcohólico, 0 = No o sin registro)
df['ALIENTO_SI'] = df['ALIENTO'].astype(str).str.strip().str.capitalize()
df['ALIENTO_SI'] = df['ALIENTO_SI'].map({'Sí': 1, 'Si': 1, 'No': 0})
# Imputamos nulos con la moda histórica (0) para mantener la estabilidad matemática
df['ALIENTO_SI'] = df['ALIENTO_SI'].fillna(0).astype(int)

#  Transformación de CINTURON (Nos interesa el riesgo: 1 = NO usaba cinturón, 0 = Sí usaba)
df['CINTURON_NO'] = df['CINTURON'].astype(str).str.strip().str.capitalize()
df['CINTURON_NO'] = df['CINTURON_NO'].map({'No': 1, 'Sí': 0, 'Si': 0, 'Se ignora': np.nan})
# Imputamos los valores "Se ignora" / NaN con la moda temporal para no perder la fila en la matriz
moda_cinturon = df['CINTURON_NO'].mode()[0] if not df['CINTURON_NO'].mode().empty else 0
df['CINTURON_NO'] = df['CINTURON_NO'].fillna(moda_cinturon).astype(int)



# ASEGURAR FORMATO NUMÉRICO DE VEHÍCULOS Y VÍCTIMAS
# Seleccionamos los tipos de vehículos clave para el análisis
columnas_vehiculos = [
    'AUTOMOVIL', 'MOTOCICLET', 'BICICLETA', 'CAMIONETA', 'CAMION', 'MICROBUS'
]

# Convertimos a entero rellenando nulos con cero (0 vehículos de ese tipo involucrados)
for col in columnas_vehiculos:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Variables de afectación física y comportamiento que ya conocemos
columnas_afectacion = ['CONDMUERTO', 'CONDHERIDO', 'PEATMUERTO', 'PEATHERIDO', 'CONDUCTOR_FUGADO']
for col in columnas_afectacion:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Aseguramos limpieza de edad y hora (con imputación temporal de edad para evitar NaN)
df['ID_HORA'] = pd.to_numeric(df['ID_HORA'], errors='coerce').fillna(0).astype(int)
df['ID_EDAD_TEMP'] = pd.to_numeric(df['ID_EDAD'], errors='coerce').replace([0, 99], np.nan)
df['ID_EDAD_TEMP'] = df['ID_EDAD_TEMP'].fillna(df['ID_EDAD_TEMP'].median())

Después de realizar la limpieza y hacer las exposiciones de la información, encontramos relaciones fuertes que podemos considerar obvias: Que un vehículo este presente muestra la tendencia a que otro tipo de vehículos no estén involucrados en un accidente (Pues estas columnas muestran una tendencia exclusiva por definición). Pero hay celdas en nuestro HeatMap que presentan fenómenos relevantes.
La más fuerte de todas es que si se presentan motocicletas en un accidente, existe la gran posibilidad de que haya heridos en el accidente, también se presentan más decesos cuando esto ocurre aunque la tendencia es mucho menor.
A pesar de ser un factor de riesgo importante, la relación entre el estado de ebriedad y la gravedad del accidente resulta inconclusa.
Además se presentan dos fenómenos curiosos: Si un conductor se encuentra en estado de ebriedad es menos probable que abandone el lugar del accidente, mientras que el no uso de un cinturón de seguridad hace más propenso que el perpetrador de uan accidente huya. A pesar de que estos fenómenos puedan verse vistos influenciados por el ruido en el dataset, tanto el proceso de limpieza para solo aplicar resultados válidos de banderas, así como la gran incidencia de casos hacen que estos sean eventos relativamente frecuentes.

In [23]:
# Consolidamos la lista de columnas que queremos correlacionar
columnas_riesgo_vehiculos = (
    ['ID_EDAD_TEMP', 'ID_HORA', 'ALIENTO_SI', 'CINTURON_NO'] + 
    columnas_vehiculos + 
    columnas_afectacion
)

# Renombramos temporalmente las columnas para que el Heatmap sea sumamente elegante y entendible
nombres_legibles = {
    'ID_EDAD_TEMP': 'Edad Conductor', 'ID_HORA': 'Hora del Día',
    'ALIENTO_SI': 'Aliento Alcohólico', 'CINTURON_NO': 'NO usaba Cinturón',
    'AUTOMOVIL': 'V: Automóvil', 'MOTOCICLET': 'V: Motocicleta',
    'BICICLETA': 'V: Bicicleta', 'CAMIONETA': 'V: Camioneta',
    'CAMION': 'V: Camión Carga', 'MICROBUS': 'V: Microbús',
    'CONDMUERTO': 'Conductor Muerto', 'CONDHERIDO': 'Conductor Herido',
    'PEATMUERTO': 'Peatón Muerto', 'PEATHERIDO': 'Peatón Herido',
    'CONDUCTOR_FUGADO': 'Conductor Fugado'
}

# Extraemos la matriz de correlación de Pearson
df_sub_corr = df[columnas_riesgo_vehiculos].rename(columns=nombres_legibles)
matriz_riesgo = df_sub_corr.corr(method='pearson')

# Generamos el mapa de calor interactivo
fig_heatmap_riesgo = px.imshow(matriz_riesgo,
                               text_auto=".2f",
                               aspect="auto",
                               title="Matriz de Correlación Avanzada: Factores de Riesgo, Tipos de Vehículo y Víctimas",
                               color_continuous_scale='RdBu_r', # Rojo positivo (riesgo compartido), Azul negativo
                               zmin=-1, zmax=1)

fig_heatmap_riesgo.update_layout(template='plotly_white', width=1000, height=800)
fig_heatmap_riesgo.show()